# 📊 E-Commerce EDA & Business Intelligence Project
### ApexPlanet Software Pvt. Ltd. — Task 2
**Dataset:** Indian E-Commerce Orders (2023–2024)  
**Tools:** Python, Pandas, Matplotlib, Seaborn, SQL  
**Objective:** Uncover patterns, trends, and actionable insights from e-commerce transactional data.

## 1️⃣ Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_palette("Set2")

df = pd.read_csv("Cleaned_EcommerceOrders.csv")
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])
print("Shape:", df.shape)
df.head()

## 2️⃣ Dataset Overview

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## 3️⃣ Descriptive Statistics & Univariate Analysis

In [ ]:
df.describe().round(2)

In [ ]:
# Key KPIs
delivered = df[df['order_status']=='Delivered']
print(f"Total Revenue:    ₹{delivered['revenue'].sum():,.2f}")
print(f"Total Profit:     ₹{delivered['profit'].sum():,.2f}")
print(f"Avg Order Value:  ₹{delivered['revenue'].mean():,.2f}")
print(f"Total Orders:     {len(delivered):,}")
print(f"Total Units Sold: {delivered['quantity'].sum():,}")
print(f"Avg Discount:     {delivered['discount'].mean()*100:.1f}%")

### 📊 Distribution of Revenue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['revenue'].clip(upper=df['revenue'].quantile(0.95)), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Revenue Distribution')
axes[0].set_xlabel('Revenue (₹)')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['profit'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Profit Distribution')
axes[1].set_xlabel('Profit (₹)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('revenue_profit_dist.png', dpi=120, bbox_inches='tight')
plt.show()

### 📦 Orders by Category

In [ ]:
cat_counts = df['category'].value_counts()
colors = sns.color_palette("Set2", len(cat_counts))
plt.figure(figsize=(9,5))
bars = plt.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, cat_counts.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30, str(val), ha='center', fontsize=11)
plt.title('Number of Orders by Category')
plt.xlabel('Category')
plt.ylabel('Order Count')
plt.tight_layout()
plt.savefig('category_orders.png', dpi=120, bbox_inches='tight')
plt.show()

### 💳 Payment Mode Distribution

In [ ]:
pm = df['payment_mode'].value_counts()
plt.figure(figsize=(7,7))
plt.pie(pm.values, labels=pm.index, autopct='%1.1f%%', startangle=140,
        colors=sns.color_palette("pastel"), wedgeprops=dict(edgecolor='white'))
plt.title('Payment Mode Distribution')
plt.tight_layout()
plt.savefig('payment_mode.png', dpi=120, bbox_inches='tight')
plt.show()

## 4️⃣ SQL Business Questions
> All queries are written in `business_queries.sql`. Below are the Python equivalents for visualization.

### 🏆 Q1: Top 5 Products by Revenue

In [ ]:
top5 = (delivered.groupby('product_name')['revenue']
        .sum().sort_values(ascending=False).head(5))

plt.figure(figsize=(10,5))
bars = plt.barh(top5.index[::-1], top5.values[::-1], color=sns.color_palette("Blues_d",5))
for bar, val in zip(bars, top5.values[::-1]):
    plt.text(bar.get_width()+500, bar.get_y()+bar.get_height()/2, f'₹{val:,.0f}', va='center', fontsize=10)
plt.title('Top 5 Products by Total Revenue')
plt.xlabel('Revenue (₹)')
plt.tight_layout()
plt.savefig('top5_products.png', dpi=120, bbox_inches='tight')
plt.show()

### 📅 Q2: Monthly Revenue Trend

In [ ]:
monthly = (delivered.groupby(['year','month'])['revenue'].sum().reset_index())
monthly['period'] = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)

plt.figure(figsize=(14,5))
for yr, grp in monthly.groupby('year'):
    plt.plot(grp['period'], grp['revenue'], marker='o', label=str(yr), linewidth=2)
plt.title('Monthly Revenue Trend (2023–2024)')
plt.xlabel('Month')
plt.ylabel('Revenue (₹)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=120, bbox_inches='tight')
plt.show()

### 💰 Q3: Most Profitable Category

In [ ]:
cat_profit = (delivered.groupby('category')
              .agg(total_revenue=('revenue','sum'), total_profit=('profit','sum'))
              .assign(margin=lambda x: x['total_profit']/x['total_revenue']*100)
              .sort_values('total_profit', ascending=False))

fig, ax1 = plt.subplots(figsize=(10,5))
x = range(len(cat_profit))
ax2 = ax1.twinx()
ax1.bar(x, cat_profit['total_profit'], color='teal', alpha=0.7, label='Total Profit')
ax2.plot(x, cat_profit['margin'], 'o-', color='orange', linewidth=2, label='Margin %')
ax1.set_xticks(x)
ax1.set_xticklabels(cat_profit.index)
ax1.set_ylabel('Total Profit (₹)')
ax2.set_ylabel('Profit Margin (%)')
ax1.set_title('Category-wise Profit & Margin')
fig.legend(loc='upper right', bbox_to_anchor=(0.88,0.88))
plt.tight_layout()
plt.savefig('category_profit.png', dpi=120, bbox_inches='tight')
plt.show()
print(cat_profit.round(2))

### 🗺️ Q5: Region-wise Performance

In [ ]:
region = (delivered.groupby('region')
          .agg(revenue=('revenue','sum'), profit=('profit','sum'), orders=('order_id','count'))
          .sort_values('revenue', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13,5))
region['revenue'].plot(kind='bar', ax=axes[0], color=sns.color_palette("Set2"), edgecolor='white')
axes[0].set_title('Revenue by Region')
axes[0].set_ylabel('Revenue (₹)')
axes[0].tick_params(axis='x', rotation=30)

region['orders'].plot(kind='bar', ax=axes[1], color=sns.color_palette("muted"), edgecolor='white')
axes[1].set_title('Orders by Region')
axes[1].set_ylabel('Order Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('region_performance.png', dpi=120, bbox_inches='tight')
plt.show()

### 🏷️ Q6: Discount Impact on Profit

In [ ]:
delivered2 = delivered.copy()
delivered2['discount_bracket'] = pd.cut(
    delivered2['discount'],
    bins=[-0.01, 0.001, 0.10, 0.20, 1.0],
    labels=['No Discount', 'Low (1-10%)', 'Medium (11-20%)', 'High (>20%)']
)
disc = delivered2.groupby('discount_bracket').agg(
    avg_profit=('profit','mean'),
    total_profit=('profit','sum'),
    orders=('order_id','count')
)

plt.figure(figsize=(9,5))
bars = plt.bar(disc.index, disc['avg_profit'],
               color=['green','yellowgreen','orange','red'], edgecolor='white')
for bar, val in zip(bars, disc['avg_profit']):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
             f'₹{val:.0f}', ha='center', fontsize=10)
plt.title('Average Profit per Order by Discount Level')
plt.xlabel('Discount Bracket')
plt.ylabel('Avg Profit (₹)')
plt.tight_layout()
plt.savefig('discount_impact.png', dpi=120, bbox_inches='tight')
plt.show()

## 5️⃣ Multivariate Analysis & Correlation

In [ ]:
# Scatter: Revenue vs Profit
plt.figure(figsize=(9,5))
for cat, grp in delivered.groupby('category'):
    plt.scatter(grp['revenue'], grp['profit'], alpha=0.4, s=20, label=cat)
plt.title('Revenue vs Profit by Category')
plt.xlabel('Revenue (₹)')
plt.ylabel('Profit (₹)')
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig('revenue_vs_profit.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Correlation Matrix
num_cols = ['quantity','unit_price','discount','revenue','profit','shipping_cost']
corr = delivered[num_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, linecolor='white')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Pairplot: Key Variables (sample)
sample = delivered[num_cols].sample(500, random_state=42)
g = sns.pairplot(sample, plot_kws={'alpha':0.3, 's':10}, diag_kws={'bins':20})
g.fig.suptitle('Pairplot of Key Numeric Variables', y=1.02, fontsize=14)
plt.savefig('pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

## 6️⃣ Key Insights & KPIs

| KPI | Value |
|-----|-------|
| Top Category (Revenue) | Electronics |
| Top Region | West |
| Best Payment Mode | UPI |
| High Discount Effect | Reduces profit significantly |
| Peak Month | Consistent growth Jan→Dec |

### 🔑 Key Business Insights:
1. **Electronics** generates the highest revenue but has moderate margin.
2. **Beauty** category has the highest profit margin %.
3. **UPI** is the most preferred payment method (35%+ orders).
4. **High discounts (>20%)** lead to negative or near-zero profit.
5. **West region** (Mumbai, Pune, Ahmedabad, Surat) dominates sales.
6. **B2B segment** has higher average order values than Retail.
7. **Returned orders** are highest in Electronics & Clothing.
